# 24 · Check D4: the D3 check repeated on the A7 land maps (post hoc choice)

Marks fixed in `PLAN.md` (Check D4, the same as D3) before this notebook ran; the code is notebook 22's. The demo
data now holds the A7 N+M land maps (`demo.land_method: context_more`), which Matt chose after seeing A7 fail its
L1 mark (a post hoc decision, recorded as such in `PLAN.md`). Answer key = the
main model's own plan run on the official maps (CDL, NWI, USPVDB) and NASA POWER on the demo's 90 m grid.

- **D4a:** precision ≥ 0.6 and F1 ≥ 0.5 for each of the three D2 questions.
- **D4b:** each answers in < 5 s including the language model, after one warm-up query (plans asked afresh,
  replies in `outputs/llm/d4/`).
- **Empty case:** the North Carolina heat question returns no places, naming the climate step and the 2-day cell.
- **Information:** the same scores for all ten page examples.

In [1]:
import json
import os
import time

import numpy as np
import pandas as pd
import torch

from askmap.aether_us import TextEncoder
from askmap.catalogue import catalogue_text
from askmap.config import PROJECT_ROOT, data_root, load_config, load_env
from askmap.demo.engine import Engine
from askmap.demo.serve import load_tiles, make_contexts, make_planner
from askmap.llm import make_client, system_prompt, translate
from askmap.plan import execute
from askmap.score import pooled_counts

load_env()
cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
AU = load_config(PROJECT_ROOT / "configs" / "aether_us.yaml")
CAP = load_config(PROJECT_ROOT / "configs" / "captions.yaml")
ROOT = data_root()
DEMO = ROOT / "demo"
RES = PROJECT_ROOT / "outputs" / "results"
meta = json.loads((DEMO / "meta.json").read_text())
LEAVES = cfg["classes"]["leaves"]
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
llm = cfg["llm"]
client = make_client(llm["base_url"])
system = system_prompt((PROJECT_ROOT / "configs" / "language.md").read_text(), catalogue_text(cfg))
T_START = time.time()
print("demo land method:", meta["land_method"], "| config:", AU["demo"]["land_method"])
if meta["land_method"] != AU["demo"]["land_method"]:
    raise RuntimeError("demo data were built with a different land method than the config names")


def build_engine(log_dir):
    def ask(question, path):
        return translate(client, llm["models"]["main"], system, question, path, llm)

    planner = make_planner(cfg, ask, log_dir)
    enc = TextEncoder(meta["encoder"], meta["encoder_kind"], device, os.environ["HF_HOME"])
    contexts = make_contexts(cfg, meta, DEMO)
    return Engine(
        load_tiles(DEMO, meta),
        LEAVES,
        enc,
        planner,
        contexts,
        meta["best"],
        CAP,
        top_share=meta["top_share"],
        not_ready=meta["not_ready"],
    )


LOG = PROJECT_ROOT / "outputs" / "llm" / "d4"
if LOG.exists() and any(LOG.iterdir()):
    raise RuntimeError(f"{LOG} already has replies; D4 must ask the language model afresh")
engine = build_engine(LOG)
t0 = time.time()
engine.search("hay fields", "north_dakota", "best")  # warm-up (not scored)
print(f"engine built and warmed up in {time.time() - t0:.0f}s; best combination = {engine.best}")

demo land method: context_more | config: context_more


/Volumes/disco2/github/climate-embeddings-crop-integrity/ask-the-map/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1029 11:21:13.279000 87291 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 8807.05it/s]

engine built and warmed up in 1s; best combination = classifier


In [2]:
def score(q, area):
    t0 = time.perf_counter()
    r = engine.search(q, area, "best")
    seconds = time.perf_counter() - t0
    plan = r["plan"]
    tiles = engine.areas[area]
    pred = engine.answer_masks(plan, area)
    truth = execute(plan, {t.name: t.leaf.astype(np.int16) for t in tiles}, engine.contexts[area])
    c = pooled_counts(pred, truth, {t.name: t.valid for t in tiles})
    if c["predicted"] != r["facts"]["shown_pixels"]:
        raise AssertionError("answer masks and the shown places differ")
    ok = c["precision"] is not None and c["f1"] is not None and c["precision"] >= 0.6 and c["f1"] >= 0.5
    return {
        "question": q,
        "area": area,
        "seconds": round(seconds, 2),
        "plan_words": r["facts"]["plan_words"],
        **c,
        "a_mark": bool(ok),
        "b_mark": seconds < 5,
    }


D4 = [
    ("wetlands next to crop fields", "north_dakota"),
    ("corn fields where July 2021 had at least 5 days above 35 C", "north_dakota"),
    ("grassland near an existing solar farm", "north_carolina"),
]
rows = [score(q, a) for q, a in D4]
d4 = pd.DataFrame(rows)
d4[
    [
        "question",
        "area",
        "seconds",
        "predicted",
        "true",
        "both",
        "precision",
        "recall",
        "f1",
        "a_mark",
        "b_mark",
    ]
]

,question,area,seconds,predicted,true,both,precision,recall,f1,a_mark,b_mark
0,wetlands next to crop fields,north_dakota,1.63,0,0,0,NaN,NaN,1.000000,False,True
1,corn fields where July 2021 had at least 5 day...,north_dakota,1.50,19152,18959,17766,0.927632,0.937075,0.932329,True,True
2,grassland near an existing solar farm,north_carolina,1.13,283,192,93,0.328622,0.484375,0.391579,False,True


In [3]:
for r in rows:
    print(f"{r['question']} ({r['area']}): {r['plan_words']}")

wetlands next to crop fields (north_dakota): Wetland, within 30 m of cropland.
corn fields where July 2021 had at least 5 days above 35 C (north_dakota): Corn, where July 2021 had at least 5 days with daily high temperature above 35 °C (NASA POWER).
grassland near an existing solar farm (north_carolina): Grassland, within 100 m of solar farm.


## The empty case: North Carolina, July 2021 heat

In [4]:
e = engine.search("corn fields where July 2021 had at least 5 days above 35 C", "north_carolina", "best")
reason = e["facts"]["empty_reason"] or ""
cells = e["facts"]["climate"][0]["cells"] if e["facts"]["climate"] else []
empty_ok = (
    e["facts"]["shown_pixels"] == 0
    and "No NASA POWER cell in this area meets" in reason
    and max(c["value"] for c in cells) == 2.0
    and "closest cell had 2" in reason
)
print(reason)
print("cell values:", [c["value"] for c in cells], "| empty-case message correct:", empty_ok)

No NASA POWER cell in this area meets "July 2021 had at least 5 days with daily high temperature above 35 °C": the closest cell had 2 against a threshold of 5 (days).
cell values: [0.0, 0.0, 2.0, 0.0] | empty-case message correct: True


## Information: all ten page examples

In [5]:
done = {(r["question"], r["area"]) for r in rows}
info = [
    score(x["question"], x["area"])
    for x in AU["demo"]["page_examples"]
    if (x["question"], x["area"]) not in done
]
allq = pd.DataFrame(rows + info)
allq[["question", "area", "seconds", "predicted", "true", "precision", "recall", "f1"]]

,question,area,seconds,predicted,true,precision,recall,f1
0,wetlands next to crop fields,north_dakota,1.63,0,0,NaN,NaN,1.000000
1,corn fields where July 2021 had at least 5 day...,north_dakota,1.50,19152,18959,0.927632,0.937075,0.932329
2,grassland near an existing solar farm,north_carolina,1.13,283,192,0.328622,0.484375,0.391579
3,corn fields where July 2021 had at least 5 day...,north_carolina,0.01,0,0,NaN,NaN,1.000000
4,cropland where spring 2022 was one of the wett...,north_dakota,2.01,48212,48214,0.945200,0.945161,0.945181
5,grassland where summer 2022 rainfall was in th...,north_dakota,2.13,21601,19397,0.820703,0.913956,0.864823
6,hay fields within 200 m of a wetland,minnesota,0.63,160,197,0.306250,0.248731,0.274510
7,soybean fields more than 300 m from any wetlan...,minnesota,1.64,13783,13853,0.856562,0.852234,0.854393
8,wetlands within 100 m of developed land,north_carolina,0.91,5385,5679,0.471681,0.447262,0.459147
9,forest within 500 m of a solar farm,georgia,0.83,601,735,0.665557,0.544218,0.598802


In [6]:
d3 = json.loads((RES / "22_d3.json").read_text())
b3 = {(q["question"], q["area"]): q for q in d3["questions"] + d3["page_examples"]}
cmp = pd.DataFrame(
    [
        {
            "question": r["question"],
            "area": r["area"],
            "D3 precision": b3[(r["question"], r["area"])]["precision"],
            "D4 precision": r["precision"],
            "D3 F1": b3[(r["question"], r["area"])]["f1"],
            "D4 F1": r["f1"],
        }
        for r in rows + info
    ]
)
verdict = {
    "questions": rows,
    "page_examples": info,
    "D4a": bool(d4.a_mark.all()),
    "D4b": bool(d4.b_mark.all()),
    "empty_case_message": empty_ok,
    "land_method": meta["land_method"],
    "minutes": round((time.time() - T_START) / 60, 1),
}
(RES / "24_d4.json").write_text(json.dumps(verdict, indent=1, default=str))
print(
    f"D4a {'PASS' if verdict['D4a'] else 'FAIL'}; D4b {'PASS' if verdict['D4b'] else 'FAIL'}; "
    f"empty-case message {'OK' if empty_ok else 'WRONG'}; {verdict['minutes']} min"
)
cmp.round(3)

D4a FAIL; D4b PASS; empty-case message OK; 0.5 min


,question,area,D3 precision,D4 precision,D3 F1,D4 F1
0,wetlands next to crop fields,north_dakota,0.362,NaN,0.421,1.000
1,corn fields where July 2021 had at least 5 day...,north_dakota,0.914,0.928,0.931,0.932
2,grassland near an existing solar farm,north_carolina,0.329,0.329,0.378,0.392
3,corn fields where July 2021 had at least 5 day...,north_carolina,NaN,NaN,1.000,1.000
4,cropland where spring 2022 was one of the wett...,north_dakota,0.938,0.945,0.944,0.945
5,grassland where summer 2022 rainfall was in th...,north_dakota,0.812,0.821,0.863,0.865
6,hay fields within 200 m of a wetland,minnesota,0.282,0.306,0.315,0.275
7,soybean fields more than 300 m from any wetlan...,minnesota,0.874,0.857,0.858,0.854
8,wetlands within 100 m of developed land,north_carolina,0.465,0.472,0.457,0.459
9,forest within 500 m of a solar farm,georgia,0.459,0.666,0.437,0.599
